# Per-Finger Touch Detection Model (5-Timestep Coordinates + Velocities LSTM)

This notebook trains a **PyTorch LSTM model** across **5 sequence timesteps** ($t=1, 2, 3, 4, 5$).

### Sequence Timestep Representation ($5 \text{ timesteps} \times 10 \text{ features}$):
Each frame timestep $k \in \{1, 2, 3, 4, 5\}$ contains 10 features:
1. **Coordinates (8 features)**: `wrist{k}_x`, `wrist{k}_y`, `mcp{k}_x`, `mcp{k}_y`, `pip{k}_x`, `pip{k}_y`, `dip{k}_x`, `dip{k}_y`
2. **Transition Velocities (2 features)**:
   - For timestep $k=1$: Prepended zero velocity vector `[0.0, 0.0]` for wrist, mcp, pip, dip.
   - For timesteps $k \in \{2, 3, 4, 5\}$: Transition velocity from step $k-1$ (`wrist{k-1}_vx/vy`, `mcp{k-1}_vx/vy`, `pip{k-1}_vx/vy`, `dip{k-1}_vx/vy`).

**Total Tensor Shape per Row**: $(N, 5, 16)$

## 1. Imports & Hyperparameters Setup

In [23]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Set random seeds for reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Hyperparameters
SEQ_LEN = 5          # 5 frame timesteps
FEATURE_DIM = 16     # 8 coordinates + 8 transition velocities per timestep
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_UNITS = 32     # Baseline hidden units
DROPOUT = 0.2         # Baseline dropout
EPOCHS = 40

# Setup device agnostic code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Load & Prepare CSV Datasets (5 Timesteps with Prepended Zero Velocity)

In [24]:
def extract_5step_features(csv_path):
    df = pd.read_csv(csv_path)
    n_samples = len(df)
    
    # Matrix of shape (N, 5, 16)
    X = np.zeros((n_samples, 5, 16), dtype=np.float32)
    
    for k in range(1, 6):
        # 8 Coordinate columns for frame k
        coord_cols = [
            f"wrist{k}_x", f"wrist{k}_y",
            f"mcp{k}_x", f"mcp{k}_y",
            f"pip{k}_x", f"pip{k}_y",
            f"dip{k}_x", f"dip{k}_y"
        ]
        coords_vals = df[coord_cols].fillna(0.0).values.astype(np.float32)
        
        # 8 Velocity columns
        if k == 1:
            # Prepended zero velocities for frame 1
            vel_vals = np.zeros((n_samples, 8), dtype=np.float32)
        else:
            v_idx = k - 1
            vel_cols = [
                f"wrist{v_idx}_vx", f"wrist{v_idx}_vy",
                f"mcp{v_idx}_vx", f"mcp{v_idx}_vy",
                f"pip{v_idx}_vx", f"pip{v_idx}_vy",
                f"dip{v_idx}_vx", f"dip{v_idx}_vy"
            ]
            vel_vals = df[vel_cols].fillna(0.0).values.astype(np.float32)
            
        step_vals = np.hstack([coords_vals, vel_vals])
        X[:, k - 1, :] = step_vals
        
    # Target label
    target_col = "touch_finger" if "touch_finger" in df.columns else "touch"
    y = df[target_col].astype(str).str.strip().str.lower().isin(["1", "true", "t", "yes", "y"]).values.astype(np.float32)
    y = y.reshape(-1, 1)
    
    return X, y

TRAIN_CSV = "./data/training_data.csv"
TEST_CSV = "./data/test_data.csv"

X_train_np, y_train_np = extract_5step_features(TRAIN_CSV)
X_test_np, y_test_np = extract_5step_features(TEST_CSV)

# PyTorch Tensors
X_train = torch.from_numpy(X_train_np).type(torch.float32)
y_train = torch.from_numpy(y_train_np).type(torch.float32)
X_test = torch.from_numpy(X_test_np).type(torch.float32)
y_test = torch.from_numpy(y_test_np).type(torch.float32)

print(f"X_train shape (5 timesteps x 16 features): {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape  (5 timesteps x 16 features): {X_test.shape},  y_test shape:  {y_test.shape}")

X_train shape (5 timesteps x 16 features): torch.Size([1791, 5, 16]), y_train shape: torch.Size([1791, 1])
X_test shape  (5 timesteps x 16 features): torch.Size([315, 5, 16]),  y_test shape:  torch.Size([315, 1])


## 3. PyTorch Dataset and DataLoader

In [25]:
class VelocitySequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = VelocitySequenceDataset(X_train, y_train)
test_dataset = VelocitySequenceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 4. Building the 5-Timestep LSTM Touch Detection Model

In [26]:
class FingerTouchLSTM(nn.Module):
    def __init__(self, input_features: int = 16, hidden_units: int = 32, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        
        # LSTM Layer processing (batch_size, seq_len=5, input_features=16)
        self.lstm = nn.LSTM(
            input_size=input_features,
            hidden_size=hidden_units,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # Fully connected classifier head for binary touch prediction
        self.classifier = nn.Sequential(
            nn.Linear(in_features=hidden_units, out_features=16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(in_features=16, out_features=1)
        )
        
    def forward(self, x):
        # Forward pass through LSTM: lstm_out shape -> (batch_size, seq_len=5, hidden_units)
        lstm_out, (hn, cn) = self.lstm(x)
        
        # Select output from final sequence timestep (t=5)
        last_timestep = lstm_out[:, -1, :]
        
        # Pass unnormalized logits to classifier head
        logits = self.classifier(last_timestep)
        return logits

# Instantiate model with 16 input features and sequence length 5
model_5step = FingerTouchLSTM(input_features=FEATURE_DIM, hidden_units=HIDDEN_UNITS, num_layers=2, dropout=DROPOUT).to(device)
print(model_5step)

FingerTouchLSTM(
  (lstm): LSTM(16, 32, num_layers=2, batch_first=True, dropout=0.2)
  (classifier): Sequential(
    (0): Linear(in_features=32, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=16, out_features=1, bias=True)
  )
)


## 5. Loss Function, Optimizer, and Accuracy Function

In [27]:
# Loss Function & Optimizer
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(params=model_5step.parameters(), lr=LEARNING_RATE)

# Accuracy Calculation Function
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc

## 6. Initial Un-trained Model Evaluation

In [28]:
model_5step.eval()
with torch.inference_mode():
    sample_X = X_train[:5].to(device)
    sample_y = y_train[:5].to(device)
    
    y_logits = model_5step(sample_X)
    y_pred_probs = torch.sigmoid(y_logits)
    y_pred_labels = torch.round(y_pred_probs)

print("Sample Raw Logits:\n", y_logits.squeeze())
print("Sample Prediction Probabilities:\n", y_pred_probs.squeeze())
print("Sample Predicted Labels:\n", y_pred_labels.squeeze())
print("True Target Labels:\n", sample_y.squeeze())

Sample Raw Logits:
 tensor([-0.2053, -0.2024, -0.2022, -0.2096, -0.2055], device='cuda:0')
Sample Prediction Probabilities:
 tensor([0.4489, 0.4496, 0.4496, 0.4478, 0.4488], device='cuda:0')
Sample Predicted Labels:
 tensor([0., 0., 0., 0., 0.], device='cuda:0')
True Target Labels:
 tensor([0., 0., 0., 1., 1.], device='cuda:0')


## 7. Training and Testing Loop (5 Timesteps)

In [30]:
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)

train_losses, test_losses = [], []
train_accuracies, test_accuracies = [], []

for epoch in range(1, EPOCHS + 1):
    # --- Training Phase ---
    model_5step.train()
    train_loss, train_acc = 0.0, 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        y_logits = model_5step(X_batch)
        loss = loss_fn(y_logits, y_batch)
        y_preds = torch.round(torch.sigmoid(y_logits))
        acc = accuracy_fn(y_batch, y_preds)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * len(X_batch)
        train_acc += (acc / 100.0) * len(X_batch)
        
    train_loss /= len(train_dataset)
    train_acc = (train_acc / len(train_dataset)) * 100
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    
    # --- Testing Phase ---
    model_5step.eval()
    test_loss, test_acc = 0.0, 0.0
    with torch.inference_mode():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            test_logits = model_5step(X_batch)
            t_loss = loss_fn(test_logits, y_batch)
            test_preds = torch.round(torch.sigmoid(test_logits))
            t_acc = accuracy_fn(y_batch, test_preds)
            
            test_loss += t_loss.item() * len(X_batch)
            test_acc += (t_acc / 100.0) * len(X_batch)
            
        test_loss /= len(test_dataset)
        test_acc = (test_acc / len(test_dataset)) * 100
        test_losses.append(test_loss)
        test_accuracies.append(test_acc)
        
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch: {epoch:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

Epoch: 01 | Train Loss: 0.2159 | Train Acc: 91.85% | Test Loss: 0.2169 | Test Acc: 91.75%
Epoch: 05 | Train Loss: 0.2169 | Train Acc: 92.29% | Test Loss: 0.2255 | Test Acc: 91.75%
Epoch: 10 | Train Loss: 0.2096 | Train Acc: 92.57% | Test Loss: 0.2189 | Test Acc: 91.11%
Epoch: 15 | Train Loss: 0.2011 | Train Acc: 92.63% | Test Loss: 0.2363 | Test Acc: 91.43%
Epoch: 20 | Train Loss: 0.1860 | Train Acc: 93.63% | Test Loss: 0.2380 | Test Acc: 90.79%
Epoch: 25 | Train Loss: 0.1914 | Train Acc: 93.24% | Test Loss: 0.2402 | Test Acc: 91.75%
Epoch: 30 | Train Loss: 0.1836 | Train Acc: 93.69% | Test Loss: 0.2433 | Test Acc: 91.11%
Epoch: 35 | Train Loss: 0.1706 | Train Acc: 93.69% | Test Loss: 0.2729 | Test Acc: 91.11%
Epoch: 40 | Train Loss: 0.1734 | Train Acc: 93.75% | Test Loss: 0.2785 | Test Acc: 91.75%


## 8. Plotting Training and Testing Curves

In [8]:
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), train_losses, label="Train Loss", color="blue")
plt.plot(range(1, EPOCHS + 1), test_losses, label="Test Loss", color="red", linestyle="--")
plt.title("Loss Curves (5 Timesteps: Coords + Velocities)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), train_accuracies, label="Train Accuracy", color="blue")
plt.plot(range(1, EPOCHS + 1), test_accuracies, label="Test Accuracy", color="red", linestyle="--")
plt.title("Accuracy Curves (5 Timesteps: Coords + Velocities)")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.legend()

plt.tight_layout()
plt.show()

## 9. Single Sequence Inference Helper Function

In [9]:
def predict_touch(model, seq_5x16):
    """
    Predicts touch event for a single 5x16 feature sequence window.
    """
    model.eval()
    tensor_in = torch.tensor(seq_5x16, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.inference_mode():
        logits = model(tensor_in)
        prob = torch.sigmoid(logits).item()
        
    is_touch = prob >= 0.5
    return prob, is_touch

# Test sample
sample_seq = X_test[0].numpy()
prob, is_touch = predict_touch(model_5step, sample_seq)
print(f"Predicted Touch Probability: {prob:.4f} -> Is Touch: {is_touch}")

## 10. Saving Model Weights

In [10]:
MODEL_SAVE_PATH = "finger_touch_5step_lstm.pth"
torch.save(model_5step.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved successfully to: {MODEL_SAVE_PATH}")